In [22]:
import joblib
import pandas as pd
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_sample_weight
import warnings

In [23]:
target_column = "health_condition"

In [24]:
X_train = pd.read_csv("../data/intermediate/train_features.csv")
X_valid = pd.read_csv("../data/intermediate/valid_features.csv")
X_test = pd.read_csv("../data/intermediate/test_features.csv")

y_train = pd.read_csv("../data/intermediate/train_labels.csv")
y_valid = pd.read_csv("../data/intermediate/valid_labels.csv")

X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 552070 entries, 0 to 552069
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   sleep_duration                491266 non-null  float64
 1   heart_rate                    545802 non-null  float64
 2   bmi                           541053 non-null  float64
 3   calorie_expenditure           509705 non-null  float64
 4   step_count                    540909 non-null  float64
 5   exercise_duration             546550 non-null  float64
 6   water_intake                  517224 non-null  float64
 7   diet_type                     546580 non-null  str    
 8   stress_level                  485727 non-null  float64
 9   sleep_quality                 505469 non-null  float64
 10  physical_activity_level       522744 non-null  float64
 11  smoking_alcohol               529184 non-null  float64
 12  gender                        535022 non-null  str    


In [25]:
X_train['diet_type'] = X_train['diet_type'].fillna('null')
X_valid['diet_type'] = X_valid['diet_type'].fillna('null')
X_test['diet_type'] = X_test['diet_type'].fillna('null')

X_train['gender'] = X_train['gender'].fillna('null')
X_valid['gender'] = X_valid['gender'].fillna('null')
X_test['gender'] = X_test['gender'].fillna('null')

In [26]:
X_train['diet_type'] = X_train['diet_type'].astype('category')
X_valid['diet_type'] = X_valid['diet_type'].astype('category')
X_test['diet_type'] = X_test['diet_type'].astype('category')

X_train['gender'] = X_train['gender'].astype('category')
X_valid['gender'] = X_valid['gender'].astype('category')
X_test['gender'] = X_test['gender'].astype('category')

In [27]:
X = pd.concat([X_train, X_valid], axis=0)
y = pd.concat([y_train, y_valid], axis=0)

In [28]:
df_submission = pd.read_csv("../data/sample_submission.csv")
df_submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                295753 non-null  int64
 1   health_condition  295753 non-null  str  
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [ ]:
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
valid_sample_weight = compute_sample_weight(class_weight="balanced", y=y_valid)
y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)

model = CatBoostClassifier(cat_features=["diet_type", "gender"], loss_function='MultiClass', n_estimators=100, learning_rate=0.1, max_depth=8, early_stopping_rounds=100)
model.fit(X_train, y_train, sample_weight=train_sample_weight, eval_set=[(X_valid, y_valid)])

joblib.dump(model, '../models/catboost_80.pkl')

0:	learn: 0.9376930	test: 0.9460749	best: 0.9460749 (0)	total: 534ms	remaining: 52.9s
1:	learn: 0.8179299	test: 0.8322127	best: 0.8322127 (1)	total: 1.25s	remaining: 1m 1s
2:	learn: 0.7237061	test: 0.7417711	best: 0.7417711 (2)	total: 1.89s	remaining: 1m 1s
3:	learn: 0.6476613	test: 0.6684181	best: 0.6684181 (3)	total: 2.47s	remaining: 59.3s
4:	learn: 0.5851355	test: 0.6078610	best: 0.6078610 (4)	total: 3.13s	remaining: 59.5s
5:	learn: 0.5336354	test: 0.5585119	best: 0.5585119 (5)	total: 3.74s	remaining: 58.6s
6:	learn: 0.4894386	test: 0.5155999	best: 0.5155999 (6)	total: 4.56s	remaining: 1m
7:	learn: 0.4519026	test: 0.4787951	best: 0.4787951 (7)	total: 5.29s	remaining: 1m
8:	learn: 0.4196821	test: 0.4473108	best: 0.4473108 (8)	total: 5.92s	remaining: 59.9s
9:	learn: 0.3922930	test: 0.4207253	best: 0.4207253 (9)	total: 6.57s	remaining: 59.1s
10:	learn: 0.3683842	test: 0.3973703	best: 0.3973703 (10)	total: 7.16s	remaining: 57.9s
11:	learn: 0.3473874	test: 0.3763620	best: 0.3763620 (11)	

['../models/catboost_80.pkl']

In [ ]:
from sklearn.metrics import balanced_accuracy_score

y_pred = model.predict(X_valid)

val_score = balanced_accuracy_score(y_valid, y_pred, sample_weight=valid_sample_weight)
print("Validation Balanced Accuracy:", val_score)

In [31]:
model = CatBoostClassifier(cat_features=["diet_type", "gender"], loss_function='MultiClass', n_estimators=100, learning_rate=0.1, max_depth=8)
model.fit(X, y, sample_weight=y_sample_weight)

joblib.dump(model, '../models/catboost_100.pkl')

0:	learn: 0.9378109	total: 802ms	remaining: 1m 19s
1:	learn: 0.8172624	total: 1.64s	remaining: 1m 20s
2:	learn: 0.7238804	total: 2.37s	remaining: 1m 16s
3:	learn: 0.6475687	total: 3.1s	remaining: 1m 14s
4:	learn: 0.5852366	total: 3.83s	remaining: 1m 12s
5:	learn: 0.5332709	total: 4.58s	remaining: 1m 11s
6:	learn: 0.4892084	total: 5.32s	remaining: 1m 10s
7:	learn: 0.4515035	total: 6s	remaining: 1m 9s
8:	learn: 0.4192165	total: 6.69s	remaining: 1m 7s
9:	learn: 0.3912808	total: 7.45s	remaining: 1m 7s
10:	learn: 0.3671337	total: 8.22s	remaining: 1m 6s
11:	learn: 0.3464253	total: 8.94s	remaining: 1m 5s
12:	learn: 0.3282725	total: 9.68s	remaining: 1m 4s
13:	learn: 0.3121722	total: 10.4s	remaining: 1m 4s
14:	learn: 0.2979325	total: 11.1s	remaining: 1m 3s
15:	learn: 0.2852399	total: 11.9s	remaining: 1m 2s
16:	learn: 0.2741916	total: 12.6s	remaining: 1m 1s
17:	learn: 0.2644382	total: 13.4s	remaining: 1m
18:	learn: 0.2558798	total: 14.1s	remaining: 1m
19:	learn: 0.2479528	total: 14.8s	remaining:

['../models/catboost_100.pkl']

In [32]:
y_pred = model.predict(X_test)

df_submission[target_column] = y_pred
df_submission[target_column] = df_submission[target_column].replace({0:'unhealthy', 1:'at-risk', 2: 'fit'})

df_submission.to_csv('../results/catboost_baseline.csv', index=False)
df_submission

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
...,...,...
295748,985836,fit
295749,985837,at-risk
295750,985838,unhealthy
295751,985839,at-risk
